# CT color mapping with RI (10012026)

Extend CT_color_mapping_09142026.pkl with the SAM_RI cell types it lacks.

The 09142026 mapping (MO/CJ/AC/XT/DR) was built in CT_Color_Mapping.ipynb from
CT_colors_09102026.csv = colorRampPalette(brewer.pal(9, 'Set1'))(309) (Rcolor.rmd).
All 309 colors are used, so the existing assignments are kept unchanged and each
missing RI type (ss_subclass_v5_nounlabeled) gets a new color chosen greedily to be
as far as possible, in CIELAB, from every color already in the mapping
(farthest-point selection over a dense RGB grid; deterministic).

In [1]:
import pickle

import h5py
import numpy as np

OLD_PKL = 'CT_color_mapping_09142026.pkl'
NEW_PKL = 'CT_color_mapping_withRI_10012026.pkl'
RI_H5AD = 'Active_SAM_joined/SAM_RI_joined_soupx_cleaned_01142026.h5ad'
RI_COL = 'ss_subclass_v5_nounlabeled'

In [2]:
def hex_to_rgb(h):
    h = h.lstrip('#')[:6]
    return np.array([int(h[i:i + 2], 16) for i in (0, 2, 4)]) / 255.0

In [3]:
def rgb_to_lab(rgb):
    rgb = np.atleast_2d(rgb)
    lin = np.where(rgb > 0.04045, ((rgb + 0.055) / 1.055) ** 2.4, rgb / 12.92)
    M = np.array([[0.4124, 0.3576, 0.1805],
                  [0.2126, 0.7152, 0.0722],
                  [0.0193, 0.1192, 0.9505]])
    xyz = lin @ M.T / np.array([0.95047, 1.0, 1.08883])
    f = np.where(xyz > 0.008856, np.cbrt(xyz), 7.787 * xyz + 16 / 116)
    return np.stack([116 * f[:, 1] - 16,
                     500 * (f[:, 0] - f[:, 1]),
                     200 * (f[:, 1] - f[:, 2])], axis=1)

In [4]:
def natural_key(s):
    # ri_2 before ri_10; numbered Allen subclasses sort by their number
    import re
    return [int(t) if t.isdigit() else t for t in re.split(r'(\d+)', s)]

In [5]:
with open(OLD_PKL, 'rb') as f:
    color_mapping = pickle.load(f)

with h5py.File(RI_H5AD, 'r') as f:
    g = f['obs'][RI_COL]
    cats = [c.decode() if isinstance(c, bytes) else c for c in g['categories'][:]]
    codes = np.unique(g['codes'][:])
ri_types = {cats[i] for i in codes if i >= 0}

missing = sorted(ri_types - set(color_mapping), key=natural_key)
print(f'{len(color_mapping)} types in {OLD_PKL}; {len(ri_types)} RI types; {len(missing)} missing')

309 types in CT_color_mapping_09142026.pkl; 152 RI types; 74 missing


In [6]:
# Candidate pool: 24-level RGB grid, limited to the lightness range of the existing palette
grid = np.linspace(0, 1, 24)
cand_rgb = np.array(np.meshgrid(grid, grid, grid, indexing='ij')).reshape(3, -1).T
cand_lab = rgb_to_lab(cand_rgb)
old_lab = rgb_to_lab(np.array([hex_to_rgb(c) for c in color_mapping.values()]))
keep = (cand_lab[:, 0] >= old_lab[:, 0].min()) & (cand_lab[:, 0] <= old_lab[:, 0].max())
cand_rgb, cand_lab = cand_rgb[keep], cand_lab[keep]

min_dist = np.min(np.linalg.norm(cand_lab[:, None, :] - old_lab[None, :, :], axis=2), axis=1)
new_colors = []
for _ in missing:
    i = int(np.argmax(min_dist))
    r, g_, b = (np.round(cand_rgb[i] * 255)).astype(int)
    new_colors.append(f'#{r:02X}{g_:02X}{b:02X}')
    min_dist = np.minimum(min_dist, np.linalg.norm(cand_lab - cand_lab[i], axis=1))

In [7]:
new_mapping = dict(color_mapping)
for ct, col in zip(missing, new_colors):
    new_mapping[ct] = col

assert all(new_mapping[k] == v for k, v in color_mapping.items())
assert ri_types <= set(new_mapping)
assert len(set(new_mapping.values())) == len(new_mapping)

In [8]:
with open(NEW_PKL, 'wb') as handle:
    pickle.dump(new_mapping, handle, protocol=pickle.HIGHEST_PROTOCOL)
print(f'wrote {NEW_PKL}: {len(new_mapping)} types ({len(missing)} added)')
for ct, col in zip(missing, new_colors):
    print(f'  {ct}\t{col}')

wrote CT_color_mapping_withRI_10012026.pkl: 383 types (74 added)
  158 MRN-PAG Nkx6-1 Glut	#8500FF
  162 CUN Evx2 Lhx2 Glut	#00FF00
  176 SCig Foxb1 Glut	#FF16DE
  177 SCig-an-PPT Foxb1 Glut	#006FFF
  182 CUN-PPN Evx2 Meis2 Glut	#FFF4BC
  191 PAG-MRN Rln3 Gaba	#00FFFF
  192 PPN-CUN-PCG Otp En1 Gaba	#FF0085
  198 IC Six3 En2 Gaba	#A6FF59
  201 PAG-RN Nkx2-2 Otx1 Gaba	#00FFB1
  221 LDT-PCG Vsx2 Lhx4 Glut	#902CC8
  224 PCG-PRNr Vsx2 Nkx6-1 Glut	#C8F4FF
  271 NI-RPO Gata3 Nr4a2 Gaba	#9B9BFF
  276 LDT-PCG St18 Gaba	#BCFFA6
  319 Astro-TE NN	#64640B
  ri_0	#F485FF
  ri_1	#00D3FF
  ri_2	#FFBCA6
  ri_3	#00BC00
  ri_4	#85A600
  ri_5	#BCC8FF
  ri_6	#BC0090
  ri_7	#B1B164
  ri_8	#00FF7A
  ri_9	#9BE9C8
  ri_10	#4E4EFF
  ri_11	#DE00FF
  ri_12	#4E59BC
  ri_13	#F4F485
  ri_14	#906F4E
  ri_15	#FFDEF4
  ri_16	#A67AFF
  ri_17	#E9B1FF
  ri_18	#009BFF
  ri_19	#FF0059
  ri_20	#00D3A6
  ri_21	#376F00
  ri_22	#C8FF00
  ri_23	#BC0B64
  ri_24	#FFC87A
  ri_25	#C8D3BC
  ri_26	#DE59FF
  ri_27	#FF5985
  ri_28	#7AF